<a href="https://colab.research.google.com/github/sp0ntanius/tcc_sumarizacao_tfidf/blob/main/Testes_extrativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rouge_score bert_score spacy pandas scikit-learn datasets
!python -m spacy download pt_core_news_lg


In [7]:
# IMPORTAÇÃO DA BASE DE DADOS

from datasets import load_dataset
import pandas as pd

# Conjunto de dados RecognaSumm
conj_dados = load_dataset("recogna-nlp/recognasumm")

# Converter a parte de treinamento para uma tabela para facilitar a visualização
table_treino = pd.DataFrame(conj_dados['train'])

# print das 5 primeiras linhas da tabela
table_treino.head()

  Preparing metadata (setup.py) ... done


In [8]:
# UTILIZAÇÃO DA TABELA DE TREINO

# Listagem das colunas
print("Colunas do conjunto de dados:")
print(table_treino.columns.tolist())
print("-" * 50)

# Exibição da primeira notícia na íntegra
not_exemplo = table_treino ['Noticia'][0]
print("Conteúdo da primeira notícia:")
print(not_exemplo)

# Exibição do sumário da primeira notícia
sum_exemplo = table_treino['Sumario'][0]
print("\nSumário da primeira notícia:")
print(sum_exemplo)

Colunas do conjunto de dados:
['index', 'Titulo', 'Subtitulo', 'Noticia', 'Categoria', 'Autor', 'Data', 'URL', 'Autor_corrigido', 'Sumario']
--------------------------------------------------
Conteúdo da primeira notícia:
A ex-BBB 23 Paula Freitas compartilhou em suas redes sociais...

Sumário da primeira notícia:
Ex-BBB Paula se emociona com novo procedimento estético: 'Desde os 16 anos que sonho em fazer isso' Ex-sister compartilhou vídeo da preparação para a cirurgia no corpo.


In [12]:
import spacy

# Baixar e carregar a base de conhecimento gramatical em português
nlp = spacy.load("pt_core_news_lg")

# Função de limpeza do texto
def pre_proc_text(texto):
    if not isinstance(texto, str):    # se o texto for nulo ou inválido, retorna vazio
        return ""

    # processamento de texto (convertendo para minúsculas)
    documento = nlp(texto.lower())

    tokens_limpos = []
    for palavra in documento: # filtro de palavras vazias, pontuações e espaço em branco
        if not palavra.is_stop and not palavra.is_punct and not palavra.is_space:
            tokens_limpos.append(palavra.lemma_)

    return " ".join(tokens_limpos) # retorno das palavras limpas em um texto

# teste de função com a primeira notícia
not_limpa = pre_proc_text(not_exemplo)

print("Texto original (500 primeiros caracteres):")
print(not_exemplo[:500] + "...\n")

print("Texto após processamento (500 primeiros caracteres):")
print(not_limpa[:500] + "...")

Texto original (500 primeiros caracteres):
A ex-BBB 23 Paula Freitas compartilhou em suas redes sociais...

Texto após processamento (500 primeiros caracteres):
ex-bbb 23 Paula freitas compartilhar rede social vídeo bastidor procedimento estético...


In [11]:
# TESTE DE CÁLUCLO DO TF-IDF I
from sklearn.feature_extraction.text import TfidfVectorizer

# dividindo a notícia original em frases matemáticas
doc_spacy = nlp(not_exemplo)
frases_orig = [sentenca.text for sentenca in doc_spacy.sents]

# limpar cada frase individualmente
frases_limp = [pre_proc_text(frase) for frase in frases_orig]

# inicializar calculadora estatística
vetorizador = TfidfVectorizer()

# construir matriz matemática com o peso de cada palavra nas frases
matriz_tfidf = vetorizador.fit_transform(frases_limp)

# resultados
print(f"A notícia foi dividida em {len(frases_orig)} frases distintas.")
print(f"Após a limpeza, o algoritmo identificou {len(vetorizador.get_feature_names_out())} palavras únicas essenciais.")

A notícia foi dividida em 14 frases distintas.
Após a limpeza, o algoritmo identificou 69 palavras únicas essenciais.


In [13]:
# TESTE DE CÁLCULO DO TF-IDF II

import numpy as np

# lista das pontuações das frases (pontuação 0, frase 0)
pont_frases = np.array(matriz_tfidf.sum(axis=1)).flatten()

# associar pontuação e a posição original com cada frase
frases_com_pont = []
for indice, pont in enumerate(pont_frases):
    frases_com_pont.append((indice, pont, frases_orig[indice]))

# ordenar frases da maior pontuação para a menor
frases_ord = sorted(frases_com_pont, key=lambda x: x[1], reverse=True)

# definir o tamanho do resumo e selecioná-las
qtdd_frases_resum = 3
melhores_frases = frases_ord[:qtdd_frases_resum]

# reordenar as frases escolhidas para a ordem cronológica que aparecem no texto
melhores_frases_cron = sorted(melhores_frases, key=lambda x:x[0])

# unir frases para o texto final
resumo_gerado = " ".join([frase[2] for frase in melhores_frases_cron])

# resultados
print("Resumo gerado pelo algoritmo:\n", resumo_gerado)
print("-" * 50)
print("Resumo de referência (gabarito humano):\n", table_treino['Sumario'][0])


Resumo gerado pelo algoritmo:
Paula Freitas compartilhou em suas redes sociais um vídeo...
--------------------------------------------------
Resumo de referência (gabarito humano):
Ex-BBB Paula se emociona com novo procedimento estético...


In [14]:
# EXIBIÇÃO DA MÉTRICA ROUGE PARA AVALIAÇÃO DO ALGORITMO SELECIONADO

from rouge_score import rouge_scorer

# isolamento do resumo de referência
resumo_ref = table_treino['Sumario'][0]

# inicialização do calculador Rouge
avaliador = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

# calculando pontuação do nosso resumo com a referência
pontuacao = avaliador.score(resumo_ref, resumo_gerado)

# resultados
print("Resultados da avaliação Rouge:\n")

for metrica, valores in pontuacao.items():
    print(f"--- {metrica.upper()} ---")
    print(f"Precision: {valores.precision * 100:.2f}%")
    print(f"Recall  : {valores.recall * 100:.2f}%")
    print(f"F1-Score: {valores.fmeasure * 100:.2f}%\n")

Resultados da avaliação Rouge:

--- ROUGE1 ---
Precision: 23.61%
Recall  : 53.12%
F1-Score: 32.69%

--- ROUGE2 ---
Precision: 5.63%
Recall  : 12.90%
F1-Score: 7.84%

--- ROUGEL ---
Precision: 13.89%
Recall  : 31.25%
F1-Score: 19.23%


In [18]:
# EXIBIÇÃO DA MÉTRICA BERTSCORE PARA AVALIAÇÃO SEMÂNTICA DO EXEMPLO ISOLADO

import evaluate

print("Carregando o modelo BERTScore...")
avaliador_bertscore = evaluate.load("bertscore")

# Calculando a pontuação semântica do nosso resumo com a referência
# IMPORTANTE: O BERTScore exige que as variáveis estejam dentro de listas [ ], mesmo sendo apenas um texto.
pontuacao_bert = avaliador_bertscore.compute(
    predictions=[resumo_gerado],
    references=[resumo_ref],
    lang="pt"
)

# Resultados
print("\nResultados da avaliação BERTScore:\n")
print("--- BERTSCORE (Análise Semântica) ---")
print(f"Precision: {pontuacao_bert['precision'][0] * 100:.2f}%")
print(f"Recall   : {pontuacao_bert['recall'][0] * 100:.2f}%")
print(f"F1-Score : {pontuacao_bert['f1'][0] * 100:.2f}%\n")

Carregando o modelo BERTScore (isso pode levar alguns segundos na primeira vez)...

Resultados da avaliação BERTScore:

--- BERTSCORE (Análise Semântica) ---
Precision: 65.98%
Recall   : 73.39%
F1-Score : 69.49%


In [15]:
# TESTE DE MUDANÇA DE FRASES SELECIONADAS NO ALGORITMO TF-IDF (ROUGE)

# redefinição do tamanho do resumo para apenas 2 frases
qtdd_frases_resumo = 2
melhores_frases = frases_ord[:qtdd_frases_resumo]

# reordenar cronologicamente e unir o texto
melhores_frases_cron = sorted(melhores_frases, key=lambda x: x[0])
resumo_gerado_2_frases = " ".join([frase[2] for frase in melhores_frases_cron])

print("Resumo gerado (2 Frases):\n", resumo_gerado_2_frases)
print("-" * 50)

# recalcular a pontuação ROUGE com o novo resumo
pontuacao_2_frases = avaliador.score(resumo_ref, resumo_gerado_2_frases)

print("Novos resultados da avaliação Rouge:\n")
for metrica, valores in pontuacao_2_frases.items():
    print(f"--- {metrica.upper()} ---")
    print(f"Precision: {valores.precision * 100:.2f}%")
    print(f"Recall   : {valores.recall * 100:.2f}%")
    print(f"F1-Score : {valores.fmeasure * 100:.2f}%\n")

Resumo gerado (2 Frases):
 Paula Freitas compartilhou em suas redes sociais um vídeo...
--------------------------------------------------
Novos resultados da avaliação Rouge:

--- ROUGE1 ---
Precision: 30.00%
Recall   : 46.88%
F1-Score : 36.59%


In [19]:
# TESTE DE MUDANÇA DE FRASES SELECIONADAS NO ALGORITMO TF-IDF (BERTSCORE)

import evaluate

# Recalcular o resumo com 2 frases para garantir a consistência das variáveis
qtdd_frases_resumo = 2
melhores_frases = frases_ord[:qtdd_frases_resumo]
melhores_frases_cron = sorted(melhores_frases, key=lambda x: x[0])
resumo_gerado_2_frases = " ".join([frase[2] for frase in melhores_frases_cron])

# Inicializar o avaliador semântico
avaliador_bertscore = evaluate.load("bertscore")

# Calcular a pontuação semântica
pontuacao_bert_2_frases = avaliador_bertscore.compute(
    predictions=[resumo_gerado_2_frases],
    references=[resumo_ref],
    lang="pt"
)

# Exibição dos resultados
print("Resumo gerado (2 Frases):\n", resumo_gerado_2_frases)
print("-" * 50)
print("Novos resultados da avaliação BERTScore:\n")
print("--- BERTSCORE (Análise Semântica) ---")
print(f"Precision: {pontuacao_bert_2_frases['precision'][0] * 100:.2f}%")
print(f"Recall   : {pontuacao_bert_2_frases['recall'][0] * 100:.2f}%")
print(f"F1-Score : {pontuacao_bert_2_frases['f1'][0] * 100:.2f}%\n")

Resumo gerado (2 Frases):
 A Yalochat, startup que oferece inteligência artificial...
--------------------------------------------------
Novos resultados da avaliação BERTScore:

--- BERTSCORE (Análise Semântica) ---
Precision: 65.98%
Recall   : 73.39%
F1-Score : 69.49%


In [20]:
# PROCESSAMENTO EM LOTE DE 100 NOTÍCIAS DO DATASET RECOGNASUMM

import numpy as np
import time
import evaluate

print("Carregando o modelo semântico do BERTScore...")
avaliador_bertscore = evaluate.load("bertscore")

print("Iniciando o processamento em lote de 100 notícias...\n")

# Listas para armazenar as métricas de cada notícia
metricas_rouge = []
metricas_bertscore = []

# Processar as primeiras 100 notícias
for i in range(100):
    # Preparar dados
    noticia = table_treino['Noticia'][i]
    sumario_ref = table_treino['Sumario'][i]
    
    # Processar notícia
    doc_spacy = nlp(noticia)
    frases_orig = [sentenca.text for sentenca in doc_spacy.sents]
    frases_limp = [pre_proc_text(frase) for frase in frases_orig]
    
    # Calcular TF-IDF
    vetorizador = TfidfVectorizer()
    matriz_tfidf = vetorizador.fit_transform(frases_limp)
    
    # Extrair frases mais importantes (2 frases)
    pont_frases = np.array(matriz_tfidf.sum(axis=1)).flatten()
    frases_com_pont = [(idx, pont, frases_orig[idx]) for idx, pont in enumerate(pont_frases)]
    frases_ord = sorted(frases_com_pont, key=lambda x: x[1], reverse=True)
    melhores_frases = frases_ord[:2]
    melhores_frases_cron = sorted(melhores_frases, key=lambda x: x[0])
    resumo_gerado = " ".join([frase[2] for frase in melhores_frases_cron])
    
    # Calcular métricas ROUGE
    avaliador_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    pontuacao_rouge = avaliador_rouge.score(sumario_ref, resumo_gerado)
    metricas_rouge.append(pontuacao_rouge)
    
    # Mostrar progresso
    if (i + 1) % 10 == 0:
        print(f"Notícias processadas: {i + 1}/100...")

print("\nProcessamento TF-IDF concluído! Calculando as notas semânticas (BERTScore)...")

Carregando o modelo semântico do BERTScore...
Iniciando o processamento em lote de 100 notícias...

Notícias processadas: 10/100...
Notícias processadas: 20/100...
Notícias processadas: 30/100...
Notícias processadas: 40/100...
Notícias processadas: 50/100...
Notícias processadas: 60/100...
Notícias processadas: 70/100...
Notícias processadas: 80/100...
Notícias processadas: 90/100...
Notícias processadas: 100/100...

Processamento TF-IDF concluído! Calculando as notas semânticas (BERTScore)...
